# 05: Rotating 3D Globe Animation (Interactive Preview & Movie)

This notebook demonstrates **interactive exploration** and **animated 360-degree rotation**:
1. Synthesizing planetary Rossby wave dynamics on a spherical mesh.
2. Interactively exploring the 3D wave pattern before animation rendering with `map_single_frame(..., sFilename_out=None)`.
3. Generating preview frames with `create_animation_preview` to inspect camera perspectives.
4. Rendering smooth rotating animations to animated GIF or MP4 with `animate_rotating_frames`.

In [ ]:
import os
from pathlib import Path
import numpy as np
import geovista as gv
from IPython.display import Image, display

import pyvista as pv

# Configure interactive 3D rendering backend in Jupyter:
# "html" embeds the complete interactive 3D WebGL scene inline (srcdoc),
# preventing VS Code / browser iframe security from blocking localhost ports.
try:
    import trame

    pv.set_jupyter_backend("html")
    print("✓ Interactive backend: in-cell 3D WebGL widget (html)")
except ImportError:
    pv.set_jupyter_backend("none")
    print("ℹ Interactive backend: native desktop 3D window")
    print("  To embed interactive 3D widgets directly inside notebook cells: pip install 'pyvista[jupyter]'")

import pyearthviz3d
from pyearthviz3d import (
    AnimationConfig,
    ScalarBarConfig,
    VisualizationConfig,
    animate_rotating_frames,
    map_single_frame,
)
from pyearthviz3d.geovista.animate_rotating_frames import create_animation_preview

print(f"pyearthviz3d version: {pyearthviz3d.__version__}")

## Step 1: Synthesize Planetary Wave Mesh

We generate an atmospheric spherical harmonic pattern representing zonal wavenumber-3 planetary waves.

In [ ]:
lons = np.linspace(-180, 180, 49)
lats = np.linspace(-90, 90, 25)
mesh = gv.Transform.from_1d(lons, lats)

lon_grid, lat_grid = np.meshgrid(
    np.radians((lons[:-1] + lons[1:]) / 2.0),
    np.radians((lats[:-1] + lats[1:]) / 2.0),
)

# Rossby wavenumber 3 planetary wave
anomaly = np.cos(3.0 * lon_grid) * np.sin(2.0 * lat_grid)
mesh.cell_data["vorticity"] = anomaly.ravel()
valid_indices = np.arange(mesh.n_cells)

print(f"Planetary wave mesh ready: {mesh.n_cells} cells.")

## Step 2: Interactive 3D Mesh Inspection

Before rendering a full 360° animation movie, launch the interactive 3D view to inspect the wave structure and check color palette / camera angles.

In [ ]:
viz_config = VisualizationConfig(
    longitude_focus=0.0,
    latitude_focus=25.0,
    zoom_factor=0.8,
    show_coastlines=True,
    show_graticule=True,
    colormap="coolwarm",
    verbose=True,
)

scalar_config = ScalarBarConfig(
    title="Planetary Wave Vorticity",
    fmt="%.2f",
    n_labels=5,
    orientation="horizontal",
)

# Interactive 3D preview view
map_single_frame(
    pMesh=mesh,
    aValid_cell_indices=valid_indices,
    pConfig=viz_config,
    sScalar="vorticity",
    sUnit="1/s",
    sFilename_out=None,  # Interactive 3D mode
    scalar_config=scalar_config,
)

## Step 3: Configure Animation & Preview Key Camera Angles

`create_animation_preview` exports snapshots of key camera angles so you can verify colorbars, lighting, and framing before rendering.

In [ ]:
anim_config = AnimationConfig(
    frames=12,    # 12 frames for quick demonstration
    speed=30.0,   # 30 degrees per frame = 360° total rotation
    framerate=6,  # 6 frames per second
)

preview_dir = "animation_previews"
preview_result = create_animation_preview(
    mesh=mesh,
    valid_cell_indices=valid_indices,
    viz_config=viz_config,
    anim_config=anim_config,
    preview_frames=4,
    scalar_name="vorticity",
    unit="1/s",
    output_dir=preview_dir,
)

preview_files = preview_result.get("generated_files", [])
print(f"Generated {len(preview_files)} preview frames:")
for pfile in preview_files:
    print(f"  • {pfile}")

# Display sample angles
if len(preview_files) >= 2:
    print("\nDisplaying preview frame 0 (0°) and frame 2 (180°):")
    display(Image(preview_files[0], width=380))
    display(Image(preview_files[2], width=380))

## Step 4: Render Full 360° Rotating Animation

We now render the complete rotating animation to an animated GIF (`rotating_globe.gif`).

In [ ]:
output_gif = "rotating_globe.gif"

def on_progress(frame_idx, total_frames, pct):
    print(f"Rendering frame {frame_idx}/{total_frames} ({pct:.0f}%)", end="\r")

anim_result = animate_rotating_frames(
    pMesh=mesh,
    aValid_cell_indices=valid_indices,
    pConfig=viz_config,
    pConfig_anima=anim_config,
    sScalar="vorticity",
    sUnit="1/s",
    sFilename_out=output_gif,
    progress_callback=on_progress,
    return_detailed_result=True,
)

print(f"\nAnimation render status: {anim_result.success}")
print(f"Output file: {output_gif} ({os.path.getsize(output_gif) // 1024} KB)")

## Step 5: Playback Rotating Animation Inline

In [ ]:
display(Image(output_gif, width=500))